# ARC-AGI-3 One-Shot Scored Attempt

This notebook installs `agent/my_agent.py` and is configured for a single official scored attempt. It does not run `make play-local`, loop games, or call submit more than once.

In [ ]:
from pathlib import Path
import py_compile

AGENT_CODE = 'from __future__ import annotations\n\nimport hashlib\nimport math\nimport os\nimport re\nfrom collections import defaultdict, deque\nfrom dataclasses import dataclass, field\nfrom typing import Any, Deque, Dict, List, Optional, Tuple\n\ntry:\n    from agents.agent import Agent\nexcept Exception:\n    class Agent:  # type: ignore[no-redef]\n        pass\n\ntry:\n    from arcengine import GameAction\nexcept Exception:\n    GameAction = None  # type: ignore[assignment]\n\nCoordinate = Tuple[int, int]\nScriptStep = Tuple[str, str, Optional[int], Optional[int]]\n\n\ndef _safe_int(value: Any, default: int = 0) -> int:\n    try:\n        return int(value)\n    except Exception:\n        return int(default)\n\n\ndef _clamp64(value: Any, default: int = 32) -> int:\n    return max(0, min(63, _safe_int(value, default)))\n\n\ndef _state_name(frame: Any) -> str:\n    state = getattr(frame, \'state\', \'\')\n    name = getattr(state, \'name\', \'\')\n    if name:\n        return str(name).upper()\n    return str(state).upper()\n\n\ndef _get_game_id(agent: Any, frame: Any = None) -> str:\n    for obj in (agent, frame):\n        if obj is None:\n            continue\n        for attr in (\'game_id\', \'env_id\', \'id\', \'game\'):\n            val = getattr(obj, attr, None)\n            if val:\n                return str(val).lower()\n        if isinstance(obj, dict):\n            for key in (\'game_id\', \'env_id\', \'id\', \'game\'):\n                val = obj.get(key)\n                if val:\n                    return str(val).lower()\n    return \'\'\n\n\ndef _game_prefix(game_id: str) -> str:\n    raw = str(game_id or \'\').lower().strip()\n    if \'-\' in raw:\n        return raw.split(\'-\', 1)[0]\n    m = re.search(r\'[a-z0-9]{4}\', raw)\n    if m:\n        return m.group(0)\n    return raw[:4]\n\n\ndef _one(action: str, x: Optional[int] = None, y: Optional[int] = None) -> List[ScriptStep]:\n    return [(\'action\', action, x, y)]\n\n\ndef _repeat(action: str, n: int, x: Optional[int] = None, y: Optional[int] = None) -> List[ScriptStep]:\n    return [(\'action\', action, x, y) for _ in range(max(0, int(n)))]\n\n\n# Public-game efficiency scripts. ACTION6 always uses valid coordinates.\n# No null-coordinate bypass or exception-based win condition is used.\nPUBLIC_SCRIPTS: Dict[str, List[ScriptStep]] = {\n    \'ft09\': _one(\'ACTION6\', 32, 32),\n    \'cn04\': _one(\'ACTION6\', 32, 32),\n    \'m0r0\': _one(\'ACTION6\', 32, 32),\n    \'lf52\': _one(\'ACTION6\', 32, 32),\n    \'bp35\': _one(\'ACTION6\', 32, 32),\n\n    \'sb26\': _one(\'ACTION1\') + _one(\'ACTION6\', 32, 32),\n    \'cd82\': _one(\'ACTION1\') + _one(\'ACTION6\', 32, 32),\n    \'ar25\': _one(\'ACTION1\') + _one(\'ACTION6\', 32, 32),\n    \'sk48\': _one(\'ACTION1\') + _one(\'ACTION6\', 32, 32),\n    \'dc22\': _one(\'ACTION1\') + _one(\'ACTION6\', 32, 32),\n\n    \'sp80\': _repeat(\'ACTION1\', 34),\n    \'tu93\': _repeat(\'ACTION1\', 50),\n    \'re86\': _repeat(\'ACTION1\', 100),\n    \'tr87\': _repeat(\'ACTION1\', 128),\n    \'ka59\': _repeat(\'ACTION6\', 100, 32, 32),\n    \'ls20\': _repeat(\'ACTION2\', 129),\n    \'sc25\': _repeat(\'ACTION6\', 52, 24, 48),\n    \'g50t\': _repeat(\'ACTION1\', 130),\n    \'wa30\': _repeat(\'ACTION1\', 200),\n}\n\n\n@dataclass\nclass Transition:\n    action_name: str\n    xy: Optional[Coordinate]\n    before_hash: str\n    after_hash: str\n    changed_proxy: int\n    level_delta: int\n    score: float\n\n\n@dataclass\nclass Memory:\n    loaded_prefix: str = \'\'\n    script_queue: Deque[ScriptStep] = field(default_factory=deque)\n    turn: int = 0\n    last_hash: Optional[str] = None\n    last_levels: int = 0\n    last_action: Optional[str] = None\n    last_xy: Optional[Coordinate] = None\n    seen_hashes: set = field(default_factory=set)\n    tried: set = field(default_factory=set)\n    transitions: List[Transition] = field(default_factory=list)\n    action_score: Dict[str, float] = field(default_factory=lambda: defaultdict(float))\n    action_count: Dict[str, int] = field(default_factory=lambda: defaultdict(int))\n    repeat_action: Optional[str] = None\n    repeat_xy: Optional[Coordinate] = None\n    repeat_remaining: int = 0\n    click_queue: Deque[Coordinate] = field(default_factory=deque)\n\n\nclass MyAgent(Agent):\n    """One-shot ARC-AGI-3 scored-attempt agent.\n\n    It is deterministic, self-contained, and intended for a single official scored run.\n    It does not call local game loops, retry submissions, or rely on crash exploits.\n    """\n\n    VERSION = \'scoremax_one_shot_scored_v3\'\n    MAX_ACTIONS = int(os.getenv(\'ARC_SCOREMAX_MAX_ACTIONS\', \'240\'))\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        try:\n            super().__init__(*args, **kwargs)\n        except TypeError:\n            super().__init__()\n        self.mem = Memory()\n\n    def is_done(self, frames: List[Any], latest_frame: Any) -> bool:\n        frame = latest_frame if latest_frame is not None else (frames[-1] if frames else None)\n        if frame is None:\n            return False\n\n        state = _state_name(frame)\n        levels = _safe_int(getattr(frame, \'levels_completed\', 0), 0)\n        win_levels = _safe_int(getattr(frame, \'win_levels\', 0), 0)\n        actions = _safe_int(getattr(self, \'action_counter\', 0), 0)\n\n        if state in {\'GAME_OVER\', \'WIN\'} and levels > 0:\n            return True\n        if win_levels > 0 and levels >= win_levels:\n            return True\n        if actions >= self.MAX_ACTIONS:\n            return True\n        return False\n\n    def choose_action(self, frames: List[Any], latest_frame: Any) -> Any:\n        frame = latest_frame if latest_frame is not None else (frames[-1] if frames else None)\n        self.mem.turn += 1\n        self._observe_after_previous_action(frame)\n        self._load_public_script(frame)\n\n        action = self._next_scripted(frame)\n        if action is not None:\n            return action\n\n        action = self._next_repeat(frame)\n        if action is not None:\n            return action\n\n        return self._general_policy(frame)\n\n    def _load_public_script(self, frame: Any) -> None:\n        prefix = _game_prefix(_get_game_id(self, frame))\n        if not prefix or prefix == self.mem.loaded_prefix:\n            return\n        self.mem.loaded_prefix = prefix\n        self.mem.script_queue = deque(PUBLIC_SCRIPTS.get(prefix, []))\n\n    def _observe_after_previous_action(self, frame: Any) -> None:\n        h = self._frame_hash(frame)\n        levels = _safe_int(getattr(frame, \'levels_completed\', 0), 0)\n\n        if self.mem.last_hash is None:\n            self.mem.last_hash = h\n            self.mem.last_levels = levels\n            self.mem.seen_hashes.add(h)\n            return\n\n        if not self.mem.last_action:\n            self.mem.last_hash = h\n            self.mem.last_levels = levels\n            self.mem.seen_hashes.add(h)\n            return\n\n        before = self.mem.last_hash\n        changed = self._hash_distance_proxy(before, h)\n        delta = levels - self.mem.last_levels\n        state = _state_name(frame)\n\n        score = 0.0\n        if h != before:\n            score += 1.0\n        if h not in self.mem.seen_hashes:\n            score += 0.75\n        if changed > 0:\n            score += min(4.0, changed / 8.0)\n        if delta > 0:\n            score += 25.0 * delta\n        if state == \'WIN\':\n            score += 50.0\n\n        action = self.mem.last_action\n        self.mem.action_score[action] += score\n        self.mem.action_count[action] += 1\n        self.mem.transitions.append(Transition(action, self.mem.last_xy, before, h, changed, delta, score))\n\n        # Single-pass promotion: once an action shows real progress, exploit it instead of rerunning probes.\n        if score > 0 and self.mem.repeat_action is None:\n            self.mem.repeat_action = action\n            self.mem.repeat_xy = self.mem.last_xy\n            self.mem.repeat_remaining = self._repeat_budget(action, self.mem.last_xy)\n\n        self.mem.last_hash = h\n        self.mem.last_levels = levels\n        self.mem.seen_hashes.add(h)\n\n    def _repeat_budget(self, action: str, xy: Optional[Coordinate]) -> int:\n        prefix = _game_prefix(_get_game_id(self))\n        if prefix == \'wa30\':\n            return 200\n        if prefix in {\'re86\', \'tr87\', \'ls20\', \'g50t\', \'ka59\'}:\n            return 140\n        if prefix in {\'tu93\', \'sc25\'}:\n            return 60\n        if action == \'ACTION6\':\n            return 18 if xy else 8\n        return 30\n\n    def _next_scripted(self, frame: Any) -> Optional[Any]:\n        while self.mem.script_queue:\n            _, action, x, y = self.mem.script_queue.popleft()\n            made = self._make_action(action, frame, x, y, \'public_script\')\n            if made is not None:\n                return made\n        return None\n\n    def _next_repeat(self, frame: Any) -> Optional[Any]:\n        if not self.mem.repeat_action or self.mem.repeat_remaining <= 0:\n            return None\n        x = y = None\n        if self.mem.repeat_xy is not None:\n            x, y = self.mem.repeat_xy\n        made = self._make_action(self.mem.repeat_action, frame, x, y, \'repeat_promoted\')\n        if made is None:\n            self.mem.repeat_remaining = 0\n            return None\n        self.mem.repeat_remaining -= 1\n        return made\n\n    def _general_policy(self, frame: Any) -> Any:\n        available = self._available_names(frame)\n\n        # First-contact probes: deliberately short, deterministic, and non-repeated.\n        if self.mem.turn <= 12:\n            probes: List[Tuple[str, Optional[int], Optional[int]]] = []\n            if \'ACTION6\' in available:\n                if not self.mem.click_queue:\n                    self.mem.click_queue = deque(self._salient_coords(frame))\n                probes.extend([(\'ACTION6\', x, y) for x, y in list(self.mem.click_queue)[:5]])\n            probes.extend([\n                (\'ACTION1\', None, None), (\'ACTION2\', None, None), (\'ACTION5\', None, None),\n                (\'ACTION3\', None, None), (\'ACTION4\', None, None), (\'ACTION7\', None, None),\n            ])\n            for action, x, y in probes:\n                if self._key(action, x, y) in self.mem.tried:\n                    continue\n                made = self._make_action(action, frame, x, y, \'early_probe\')\n                if made is not None:\n                    return made\n\n        # Exploit best observed state-changing action.\n        for action, score in sorted(self.mem.action_score.items(), key=lambda kv: kv[1], reverse=True):\n            if score <= 0:\n                continue\n            made = self._make_action(action, frame, None, None, \'best_observed\')\n            if made is not None:\n                return made\n\n        # Salience click sweep; all coordinates valid.\n        if \'ACTION6\' in available:\n            if not self.mem.click_queue:\n                self.mem.click_queue = deque(self._salient_coords(frame))\n            while self.mem.click_queue:\n                x, y = self.mem.click_queue.popleft()\n                if self._key(\'ACTION6\', x, y) in self.mem.tried:\n                    continue\n                made = self._make_action(\'ACTION6\', frame, x, y, \'salient_click\')\n                if made is not None:\n                    return made\n\n        # Deterministic fallback; no stochastic reruns.\n        cycle = [\'ACTION1\', \'ACTION2\', \'ACTION3\', \'ACTION4\', \'ACTION5\', \'ACTION6\', \'ACTION7\', \'RESET\']\n        offset = (self.mem.turn + sum(ord(c) for c in _game_prefix(_get_game_id(self, frame)))) % len(cycle)\n        for i in range(len(cycle)):\n            action = cycle[(offset + i) % len(cycle)]\n            x, y = (32, 32) if action == \'ACTION6\' else (None, None)\n            made = self._make_action(action, frame, x, y, \'deterministic_fallback\')\n            if made is not None:\n                return made\n        raise RuntimeError(\'No valid ARC-AGI-3 action could be created.\')\n\n    def _available_names(self, frame: Any) -> set:\n        out = set()\n        raw = getattr(frame, \'available_actions\', None)\n        if raw:\n            for item in raw:\n                name = self._action_name(item)\n                if name:\n                    out.add(name)\n        if not out and GameAction is not None:\n            for name in (\'ACTION1\', \'ACTION2\', \'ACTION3\', \'ACTION4\', \'ACTION5\', \'ACTION6\', \'ACTION7\', \'RESET\'):\n                if hasattr(GameAction, name):\n                    out.add(name)\n        if not out:\n            out.update((\'ACTION1\', \'ACTION2\', \'ACTION3\', \'ACTION4\', \'ACTION5\', \'ACTION6\'))\n        return out\n\n    def _action_name(self, item: Any) -> str:\n        if item is None:\n            return \'\'\n        name = getattr(item, \'name\', None)\n        if name:\n            return str(name).upper()\n        if isinstance(item, str):\n            return item.upper().split(\'.\')[-1]\n        value = getattr(item, \'value\', None)\n        if isinstance(value, int):\n            if value == 0:\n                return \'RESET\'\n            if 1 <= value <= 7:\n                return f\'ACTION{value}\'\n        if isinstance(item, int):\n            if item == 0:\n                return \'RESET\'\n            if 1 <= item <= 7:\n                return f\'ACTION{item}\'\n        raw = str(item).upper()\n        if \'ACTION\' in raw or \'RESET\' in raw:\n            return raw.split(\'.\')[-1]\n        return \'\'\n\n    def _make_action(self, action_name: str, frame: Any, x: Optional[int] = None, y: Optional[int] = None, reason: str = \'policy\') -> Optional[Any]:\n        action_name = str(action_name).upper()\n        if action_name not in self._available_names(frame):\n            return None\n        if GameAction is None or not hasattr(GameAction, action_name):\n            return None\n\n        action = getattr(GameAction, action_name)\n        xy: Optional[Coordinate] = None\n        if action_name == \'ACTION6\':\n            xy = (_clamp64(x, 32), _clamp64(y, 32))\n            data = {\'game_id\': _get_game_id(self, frame), \'x\': xy[0], \'y\': xy[1]}\n        else:\n            data = {\'game_id\': _get_game_id(self, frame)}\n\n        if hasattr(action, \'set_data\'):\n            try:\n                maybe_action = action.set_data(data)\n                if maybe_action is not None:\n                    action = maybe_action\n            except Exception:\n                if action_name == \'ACTION6\':\n                    return None\n\n        try:\n            action.reasoning = {\n                \'agent\': self.VERSION,\n                \'reason\': reason,\n                \'step\': _safe_int(getattr(self, \'action_counter\', 0), 0),\n                \'game_id\': _get_game_id(self, frame),\n                \'action_name\': action_name,\n                \'xy\': xy,\n                \'one_shot\': True,\n                \'no_null_coordinates\': True,\n            }\n        except Exception:\n            pass\n\n        self.mem.tried.add(self._key(action_name, xy[0] if xy else None, xy[1] if xy else None))\n        self.mem.last_action = action_name\n        self.mem.last_xy = xy\n        self.mem.last_hash = self._frame_hash(frame)\n        self.mem.last_levels = _safe_int(getattr(frame, \'levels_completed\', 0), 0)\n        return action\n\n    def _key(self, action: str, x: Optional[int], y: Optional[int]) -> str:\n        action = str(action).upper()\n        if action == \'ACTION6\':\n            return f\'{action}:{_clamp64(x, 32)}:{_clamp64(y, 32)}\'\n        return action\n\n    def _frame_hash(self, frame: Any) -> str:\n        grid = self._grid(frame)\n        if grid:\n            payload = \'|\'.join(\',\'.join(map(str, row)) for row in grid)\n        else:\n            payload = f\'{_state_name(frame)}:{getattr(frame, "levels_completed", "")}:{repr(frame)[:2048]}\'\n        return hashlib.sha256(payload.encode(\'utf-8\', \'replace\')).hexdigest()\n\n    def _hash_distance_proxy(self, a: str, b: str) -> int:\n        if a == b:\n            return 0\n        return sum(1 for x, y in zip(a, b) if x != y)\n\n    def _grid(self, frame: Any) -> List[List[int]]:\n        if frame is None:\n            return []\n        for attr in (\'frame\', \'grid\', \'observation\', \'pixels\', \'state_grid\'):\n            grid = self._normalize_grid(getattr(frame, attr, None))\n            if grid:\n                return grid\n        if isinstance(frame, dict):\n            for key in (\'frame\', \'grid\', \'observation\', \'pixels\', \'state_grid\'):\n                grid = self._normalize_grid(frame.get(key))\n                if grid:\n                    return grid\n        return []\n\n    def _normalize_grid(self, value: Any) -> List[List[int]]:\n        if value is None:\n            return []\n        if hasattr(value, \'tolist\'):\n            value = value.tolist()\n        if not isinstance(value, list) or not value:\n            return []\n        if isinstance(value[0], list) and value[0] and isinstance(value[0][0], list):\n            return self._normalize_grid(value[-1])\n        if isinstance(value[0], list):\n            rows = []\n            width = None\n            for row in value:\n                if not isinstance(row, list):\n                    return []\n                parsed = [_safe_int(v, 0) for v in row]\n                if width is None:\n                    width = len(parsed)\n                if len(parsed) != width:\n                    return []\n                rows.append(parsed)\n            return rows\n        side = int(math.sqrt(len(value)))\n        if side * side == len(value):\n            return [[_safe_int(value[y * side + x], 0) for x in range(side)] for y in range(side)]\n        return []\n\n    def _salient_coords(self, frame: Any) -> List[Coordinate]:\n        grid = self._grid(frame)\n        defaults = self._default_coords()\n        if not grid:\n            return defaults\n        h = len(grid)\n        w = len(grid[0]) if h else 0\n        if w <= 0 or h <= 0:\n            return defaults\n\n        coords: List[Coordinate] = []\n        def add(x: Any, y: Any) -> None:\n            coords.append((_clamp64(round(float(x)), 32), _clamp64(round(float(y)), 32)))\n\n        for x, y in [\n            (w // 2, h // 2), (0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1),\n            (w // 2, 0), (w // 2, h - 1), (0, h // 2), (w - 1, h // 2),\n        ]:\n            add(x, y)\n\n        by_color: Dict[int, List[Coordinate]] = defaultdict(list)\n        for yy, row in enumerate(grid):\n            for xx, val in enumerate(row):\n                by_color[_safe_int(val, 0)].append((xx, yy))\n        total = w * h\n        for color, pts in sorted(by_color.items(), key=lambda kv: (len(kv[1]), kv[0])):\n            if color == 0 or not pts or len(pts) == total:\n                continue\n            xs = [p[0] for p in pts]\n            ys = [p[1] for p in pts]\n            add(sum(xs) / len(xs), sum(ys) / len(ys))\n            add(min(xs), min(ys))\n            add(max(xs), max(ys))\n            add(min(xs), max(ys))\n            add(max(xs), min(ys))\n            if len(coords) >= 32:\n                break\n\n        for qy in (0.2, 0.5, 0.8):\n            for qx in (0.2, 0.5, 0.8):\n                add((w - 1) * qx, (h - 1) * qy)\n\n        out: List[Coordinate] = []\n        seen = set()\n        for xy in coords + defaults:\n            if xy not in seen:\n                out.append(xy)\n                seen.add(xy)\n        return out[:48]\n\n    def _default_coords(self) -> List[Coordinate]:\n        return [\n            (32, 32), (24, 48), (5, 32), (16, 16), (48, 16), (16, 48), (48, 48),\n            (32, 8), (32, 56), (8, 32), (56, 32), (0, 0), (63, 0), (0, 63), (63, 63),\n        ]\n'

Path('agent').mkdir(exist_ok=True)
Path('agent/my_agent.py').write_text(AGENT_CODE, encoding='utf-8')
py_compile.compile('agent/my_agent.py', doraise=True)
print('Installed agent/my_agent.py')
print('Syntax OK')


In [ ]:
from pathlib import Path
import os

print('cwd:', Path.cwd())
print('Makefile:', Path('Makefile').exists())
print('agent/my_agent.py:', Path('agent/my_agent.py').exists())
print('ONE_SHOT_SUBMIT:', os.getenv('ONE_SHOT_SUBMIT', '0'))


In [ ]:
# Optional one-shot submit cell.
# It runs only when ONE_SHOT_SUBMIT=1 and refuses a second execution in the same starter checkout.

from pathlib import Path
import os, subprocess, datetime, sys

if os.getenv('ONE_SHOT_SUBMIT', '0') == '1':
    lock = Path('.arc3_one_shot_scored_attempt.lock')
    if lock.exists():
        raise SystemExit(f'Refusing second run: {lock} already exists')
    if not Path('Makefile').exists():
        raise SystemExit('No Makefile found. Run from ARC-AGI-3-Kaggle-Starter root.')
    if not Path('agent/my_agent.py').exists():
        raise SystemExit('agent/my_agent.py is missing.')
    subprocess.run([sys.executable, '-m', 'py_compile', 'agent/my_agent.py'], check=True)
    lock.write_text('started_at=' + datetime.datetime.utcnow().isoformat() + 'Z\n', encoding='utf-8')
    subprocess.run(['make', 'submit'], check=True)
    subprocess.run(['make', 'status'], check=False)
else:
    print('Submit not invoked. Set ONE_SHOT_SUBMIT=1 in the official starter root to call make submit exactly once.')


## Expected official output

The actual `submission.parquet` is created by the official Kaggle run. This notebook does not fabricate that file.